# 02 — Preprocessing

#Takes the raw aligned tables from notebook 01 and produces analysis-ready matrices.

#Steps:
#1. Filter low-prevalence ASVs
#2. Roll up ASVs to genus level
#3. CLR transform (compositional data handling)

#Outputs:
#- genus_counts.csv  — genus-level counts, for descriptive plots
#- genus_clr.csv     — CLR-transformed, for all statistics and ML

In [1]:
import pandas as pd
import numpy as np
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from config import PARAMS, PATHS

PROCESSED = "../data/processed/"

# Load what notebook 01 produced
asv     = pd.read_csv(PROCESSED + "feature_table.csv", index_col="sample_id")
meta    = pd.read_csv(PROCESSED + "metadata.csv",      index_col="sample_id")
tax     = pd.read_csv(PROCESSED + "taxonomy.csv",      index_col="asv_id")

print(f"ASV table: {asv.shape}")
print(f"Metadata:  {meta.shape}")
print(f"Taxonomy:  {tax.shape}")
print(f"\nDisease split:\n{meta['disease_status'].value_counts()}")

ASV table: (2238, 39868)
Metadata:  (2238, 6)
Taxonomy:  (39868, 8)

Disease split:
disease_status
HC    1204
RA    1034
Name: count, dtype: int64


## Step 1 — Roll up ASVs to genus level

Sum counts across all ASVs belonging to the same genus.

Done BEFORE filtering: a genus may consist of many individually rare ASVs
that are collectively common. Filtering at ASV level first would discard it.

ASVs with unassigned genus ("Unknown", "uncultured") are dropped — they
can't be interpreted biologically or compared against published findings.

In [2]:
tax.head()

,Kingdom,Phylum,Class,Order,Family,Genus,Species,confidence
asv_id,,,,,,,,
0000ed2ed0bc10264c79eca46bcf59bb,Bacteria,Actinobacteria,Actinobacteria,Actinomycetales,Actinomycetaceae,Actinomyces,Unknown,1.000000
000107249b95c7ee9c8003651daf33a9,Bacteria,Firmicutes,Clostridia,Clostridiales,Ruminococcaceae,Faecalibacterium,Unknown,0.999884
0001103c8c21244e4752e992a2c8c1e6,Bacteria,Firmicutes,Tissierellia,Tissierellales,Peptoniphilaceae,Anaerococcus,Unknown,1.000000
00011f9bc60eb25bfbb8f67477c224e3,Bacteria,Proteobacteria,Gammaproteobacteria,Enterobacteriales,Enterobacteriaceae,Escherichia-Shigella,Unknown,0.993301
000506b268139e230ad57cf28b42dcb0,Bacteria,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1.000000


In [3]:
# Pull the genus column from the taxonomy table
asv_to_genus = tax["Genus"]

# Mark ASVs with no usable genus assignment
unknown_mask = asv_to_genus.isin(["Unknown", "uncultured"])

# ASV IDs we're keeping (~ flips the mask)
keep_asvs = asv_to_genus[~unknown_mask].index

# Subset the ASV table to those columns
asv_known = asv[keep_asvs]

# Transpose → group by genus → sum → transpose back
genus_counts = asv_known.T.groupby(asv_to_genus[keep_asvs]).sum().T
genus_counts.index.name = "sample_id"

print(f"ASVs dropped: {unknown_mask.sum()}")
print(f"ASVs kept:    {(~unknown_mask).sum()}")
print(f"After rollup: {genus_counts.shape}")
print(f"\nReads per sample (was 10000):")
print(genus_counts.sum(axis=1).describe().round(0))

ASVs dropped: 5702
ASVs kept:    34166
After rollup: (2238, 445)

Reads per sample (was 10000):
count     2238.0
mean      9612.0
std        309.0
min       5588.0
25%       9505.0
50%       9677.0
75%       9801.0
max      10000.0
dtype: float64


In [4]:
print(genus_counts.dtypes.value_counts())

int64    445
Name: count, dtype: int64


In [5]:
# Prevalence = fraction of samples where a genus is present (count > 0)
prevalence = (genus_counts > 0).mean(axis=0)

# Keep genera above the threshold set in config.py
keep_genera = prevalence >= PARAMS["min_prevalence"]

# Subset: all rows, only the columns that passed
genus_filtered = genus_counts.loc[:, keep_genera]

print(f"Threshold: present in >= {PARAMS['min_prevalence']:.0%} of samples")
print(f"Genera before: {genus_counts.shape[1]}")
print(f"Genera after:  {genus_filtered.shape[1]}")
print(f"Dropped:       {(~keep_genera).sum()}")

print(f"\nReads per sample after filtering:")
print(genus_filtered.sum(axis=1).describe().round(0))

print(f"\nPrevalence of the 10 most widespread genera:")
print(prevalence.sort_values(ascending=False).head(10).round(3))

Threshold: present in >= 5% of samples
Genera before: 445
Genera after:  128
Dropped:       317

Reads per sample after filtering:
count     2238.0
mean      9530.0
std        389.0
min       4572.0
25%       9415.0
50%       9621.0
75%       9764.0
max      10000.0
dtype: float64

Prevalence of the 10 most widespread genera:
Genus
Bacteroides                    0.994
Blautia                        0.979
Faecalibacterium               0.958
Streptococcus                  0.952
Escherichia-Shigella           0.944
Bifidobacterium                0.932
Lachnoclostridium              0.904
Erysipelotrichaceae UCG-003    0.881
Roseburia                      0.869
Butyricicoccus                 0.863
dtype: float64


In [6]:
from skbio.stats.composition import clr, multiplicative_replacement

# 1. Replace zeros with small positive values
genus_no_zeros = multiplicative_replacement(genus_filtered.values)

# 2. Apply the centred log-ratio transform
clr_values = clr(genus_no_zeros)

# 3. Rebuild as a DataFrame with original labels
genus_clr = pd.DataFrame(
    clr_values,
    index=genus_filtered.index,
    columns=genus_filtered.columns
)

# ── Checks ───────────────────────────────────────────────
print(f"CLR matrix shape: {genus_clr.shape}")
print(f"Overall mean (should be ~0): {genus_clr.values.mean():.6f}")

print(f"\nRow sums (should all be ~0):")
print(genus_clr.sum(axis=1).describe().round(6))

print(f"\nInfinities: {np.isinf(genus_clr.values).sum()}")
print(f"NaNs:       {genus_clr.isna().sum().sum()}")

print(f"\nValue range: {genus_clr.values.min():.2f} to {genus_clr.values.max():.2f}")

# ── Save both outputs ────────────────────────────────────
genus_filtered.to_csv(PROCESSED + "genus_counts.csv")
genus_clr.to_csv(PROCESSED + "genus_clr.csv")

print(f"\nSaved genus_counts.csv {genus_filtered.shape}")
print(f"Saved genus_clr.csv    {genus_clr.shape}")

ImportError: cannot import name 'multiplicative_replacement' from 'skbio.stats.composition' (/opt/anaconda3/envs/ra-microbiome/lib/python3.10/site-packages/skbio/stats/composition/__init__.py)

In [7]:
import skbio
print("skbio version:", skbio.__version__)

import skbio.stats.composition as comp
print("\nAvailable functions:")
print([f for f in dir(comp) if not f.startswith("_")])

skbio version: 0.7.3

Available functions:
['alr', 'alr_inv', 'ancom', 'ancombc', 'centralize', 'closure', 'clr', 'clr_inv', 'dirmult_lme', 'dirmult_ttest', 'ilr', 'ilr_inv', 'inner', 'multi_replace', 'pairwise_vlr', 'perturb', 'perturb_inv', 'power', 'rclr', 'sbp_basis', 'struc_zero', 'tree_basis', 'vlr']


In [8]:
from skbio.stats.composition import clr, multi_replace

# 1. Replace zeros with small positive values
genus_no_zeros = multi_replace(genus_filtered.values)

# 2. Apply the centred log-ratio transform
clr_values = clr(genus_no_zeros)

# 3. Rebuild as a DataFrame with original labels
genus_clr = pd.DataFrame(
    clr_values,
    index=genus_filtered.index,
    columns=genus_filtered.columns
)

# ── Checks ───────────────────────────────────────────────
print(f"CLR matrix shape: {genus_clr.shape}")
print(f"Overall mean (should be ~0): {genus_clr.values.mean():.6f}")

print(f"\nRow sums (should all be ~0):")
print(genus_clr.sum(axis=1).describe().round(6))

print(f"\nInfinities: {np.isinf(genus_clr.values).sum()}")
print(f"NaNs:       {genus_clr.isna().sum().sum()}")

print(f"\nValue range: {genus_clr.values.min():.2f} to {genus_clr.values.max():.2f}")

# ── Save both outputs ────────────────────────────────────
genus_filtered.to_csv(PROCESSED + "genus_counts.csv")
genus_clr.to_csv(PROCESSED + "genus_clr.csv")

print(f"\nSaved genus_counts.csv {genus_filtered.shape}")
print(f"Saved genus_clr.csv    {genus_clr.shape}")

CLR matrix shape: (2238, 128)
Overall mean (should be ~0): 0.000000

Row sums (should all be ~0):
count    2238.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64

Infinities: 0
NaNs:       0

Value range: -2.74 to 9.50

Saved genus_counts.csv (2238, 128)
Saved genus_clr.csv    (2238, 128)
